In [ ]:
# --- paths come from human/config.py (auto-inserted by fix_notebooks.py) ---
import sys; sys.path.append('..')
from config import HUMAN_BASE


# Figure 3 (BMMC) — single-cell hematopoiesis, parallel to GTEx

Replicates the three GTEx Figure 3 panels on **single-cell BMMC** (Granja 2019), over the **same Lambert TF list** as GTEx, using **BMMC's own state-caller** (yeast GMM-AICc + KDE permutation, copied verbatim from `build_setia_input`).

Mapping to GTEx:
- GTEx **tissue** (e.g. Brain__Cortex)  ->  BMMC **cell type** (e.g. 03_Late.Eryth)
- GTEx **donor sample** within a tissue  ->  BMMC **pseudocell** within a cell type
- GTEx **organ group** (Brain = cortex + ...)  ->  BMMC **sister-subtype group** (Monocyte = CD14.Mono.1/.2 + CD16.Mono)

Panels:
- **Pie**: n_states distribution over expressed Lambert TFs
- **Consistency violin**: 56 SETIA TFs vs other TFs (n_states>=2), + a printed n_states-matched check (no ranking, no extra figure)
- **Hamming coherence**: within/between Hamming ratio per sister-subtype group + shuffled Control

Notes:
- Two adaptations that do NOT change the state-calling math: per-gene GMM plots are off (`MAKE_GMM_PLOTS=False`); the hard-coded `n_permutations=2000` in the Welch merge is exposed as `KDE_N_PERM`.
- BMMC expresses fewer TFs than GTEx (single tissue). We start from the same Lambert "Yes" list and report both denominators honestly.


## Part A -- Imports, parameters, and BMMC state-caller (verbatim)

In [ ]:
import os, math, csv, itertools, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from collections import defaultdict, Counter
from itertools import combinations
from scipy.stats import ttest_ind, gaussian_kde, mannwhitneyu, spearmanr
from scipy.spatial.distance import pdist, squareform
from statsmodels.stats.multitest import multipletests
import anndata as ad
from scipy.sparse import issparse, csr_matrix
import openpyxl

warnings.filterwarnings('ignore')
np.seterr(all='ignore')

# ===== LOCAL PATHS -- EDIT THESE =====
H5AD_PATH      = f"{HUMAN_BASE}/GTEx_v11/Granja2019_annotated.h5ad"
LAMBERT_XLSX   = f"{HUMAN_BASE}/GTEx_v11/1-s2.0-S0092867418301065-mmc2.xlsx"
SETIA56_PATH   = f"{HUMAN_BASE}/BMMC_gene_list.txt"   # the 56 SETIA TFs, one per line
CELLTYPE_COL   = 'BioClassification'
OUTDIR         = 'bmmc_tf_output'
RESULT_DIR     = './result_bmmc'                                # caller writes Steady_state_count.txt here

# ===== KNOBS =====
KDE_N_PERM      = 2000     # permutations in Welch-merge KDE test. Lower (e.g. 500) to speed up the genome-wide pass.
MAKE_GMM_PLOTS  = False    # per-gene GMM diagnostic jpgs. Keep False for the 1171-gene run.
P_VALUE_CUTOFF  = 0.01     # same as build_setia_input
USE_LOG2        = True     # state-calling on log2(CPM+1), like build_setia_input

# ===== PSEUDOCELL PARAMS (BMMC's own -- do NOT change) =====
MAX_PSEUDOCELLS_PER_CELLTYPE = 10
MIN_CELLS_PER_PSEUDOCELL     = 50
MIN_PSEUDOCELLS              = 2
RANDOM_SEED                  = 42

# ===== sister-subtype groups (organ <-> group analog) =====
SISTER_GROUPS = {
    'Erythroid':  ['02_Early.Eryth', '03_Late.Eryth'],
    'Monocyte':   ['11_CD14.Mono.1', '12_CD14.Mono.2', '13_CD16.Mono'],
    'GMP':        ['07_GMP', '08_GMP.Neut'],
    'CLP':        ['06_CLP.1', '15_CLP.2'],
    'DC':         ['09_pDC', '10_cDC'],
    'B':          ['16_Pre.B', '17_B'],
    'NaiveT_CD4': ['19_CD8.N', '20_CD4.N1', '21_CD4.N2', '22_CD4.M'],
    'Cytotoxic':  ['23_CD8.EM', '24_CD8.CM', '25_NK'],
}
# cell types to drop entirely from the analysis (unannotated)
EXCLUDE_CELLTYPES = ['14_Unk', '26_Unk']

os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(os.path.join(RESULT_DIR, 'GMM_figures', 'AIC'), exist_ok=True)
_SCF = os.path.join(RESULT_DIR, 'Steady_state_count.txt')
if os.path.exists(_SCF): os.remove(_SCF)
print('Setup OK')

### A1. `kde_likelihood_empirical_p` -- verbatim

In [ ]:
def kde_likelihood_empirical_p(A, B, bw_method=None, n_permutations=2000,
                               alternative='greater', eps=1e-300,
                               random_seed=None, return_details=False):
    if random_seed is not None:
        np.random.seed(random_seed)
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    if n_permutations < 1:
        raise ValueError("n_permutations must be >= 1")
    nA = len(A); nB = len(B)
    if nA < 2:
        raise ValueError("A must contain at least 2 points for KDE")
    def log_geo_mean_for_split(Atrain, Btest):
        kde = gaussian_kde(Atrain, bw_method=bw_method)
        dens = kde(Btest)
        logdens = np.log(dens + eps)
        return float(np.mean(logdens)), np.exp(np.mean(logdens))
    obs_loggm, obs_gm = log_geo_mean_for_split(A, B)
    combined = np.concatenate([A, B])
    perm_loggms = np.empty(n_permutations, dtype=float)
    for i in range(n_permutations):
        perm = np.random.permutation(combined)
        Aperm = perm[:nA]; Bperm = perm[nA:]
        lgm, _ = log_geo_mean_for_split(Aperm, Bperm)
        perm_loggms[i] = lgm
    if alternative == 'greater':
        p_emp = (np.sum(perm_loggms >= obs_loggm) + 1) / (n_permutations + 1)
    elif alternative == 'less':
        p_emp = (np.sum(perm_loggms <= obs_loggm) + 1) / (n_permutations + 1)
    elif alternative == 'two-sided':
        greater = (np.sum(perm_loggms >= obs_loggm) + 1) / (n_permutations + 1)
        less    = (np.sum(perm_loggms <= obs_loggm) + 1) / (n_permutations + 1)
        p_emp = 2.0 * min(greater, less); p_emp = min(p_emp, 1.0)
    else:
        raise ValueError("alternative must be 'greater', 'less', or 'two-sided'")
    if return_details:
        return {'p_emp': float(p_emp), 'obs_loggm': float(obs_loggm), 'obs_gm': float(obs_gm),
                'perm_loggms': perm_loggms, 'perm_gms': np.exp(perm_loggms)}
    return float(1-p_emp)

### A2. `teset_and_merge_welch` -- verbatim (only `2000` -> `KDE_N_PERM`)

In [ ]:
def teset_and_merge_welch(optimal_clusters, optimal_mapping, replicate_variability=None, alpha=P_VALUE_CUTOFF):
    def avg_pairwise_dist(sub):
        arr = np.asarray(sub, dtype=float); arr = arr[~np.isnan(arr)]
        if arr.size < 2: return 0.0
        diffs = np.abs(arr[:, None] - arr)
        triu = diffs[np.triu_indices(arr.size, k=1)]
        return float(np.mean(triu))
    merged = [list(sub) for sub in optimal_clusters]
    try:
        merged_map = [list(optimal_mapping[i]) for i in range(len(optimal_clusters))]
    except Exception:
        keys_sorted = sorted(optimal_mapping.keys())
        merged_map = [list(optimal_mapping[k]) for k in keys_sorted]
        if len(merged_map) < len(merged):
            start = max(keys_sorted) + 1 if keys_sorted else 0
            for idx in range(len(merged_map), len(merged)):
                merged_map.append([start + (idx - len(merged_map))])
    within_dists = [avg_pairwise_dist(sub) for sub in merged]
    mean_within = float(np.mean(within_dists)) if within_dists else 0.0
    while True:
        n = len(merged)
        if n <= 1: break
        p_mat = np.full((n, n), -np.inf, dtype=float)
        for i in range(n):
            for j in range(i + 1, n):
                a = np.asarray(merged[i], dtype=float); a = a[~np.isnan(a)]
                b = np.asarray(merged[j], dtype=float); b = b[~np.isnan(b)]
                if a.size < 2 or b.size < 2: continue
                try:
                    t_stat, p_val = ttest_ind(a, b, equal_var=False)
                    if len(a) < len(b):
                        gm = kde_likelihood_empirical_p(b, a, bw_method='scott', n_permutations=KDE_N_PERM, alternative='greater', random_seed=0)
                    else:
                        gm = kde_likelihood_empirical_p(a, b, bw_method='scott', n_permutations=KDE_N_PERM, alternative='greater', random_seed=0)
                    p_val = max(p_val, gm)
                except Exception:
                    continue
                if np.isnan(p_val): continue
                p_mat[i, j] = p_val; p_mat[j, i] = p_val
        max_p = np.max(p_mat)
        if not np.isfinite(max_p) or max_p < alpha: break
        flat_idx = np.argmax(p_mat); i, j = divmod(flat_idx, n)
        if i == j: break
        if i > j: i, j = j, i
        merged[i].extend(merged[j]); merged_map[i].extend(merged_map[j])
        del merged[j]; del merged_map[j]
    merged_mapping = {k: merged_map[k] for k in range(len(merged_map))}
    return merged, merged_mapping

### A3. Helper functions -- verbatim

In [ ]:
def find_elbow_idx_by_cutoff(seq, cutoff):
    if len(seq) < 2: return None
    freq = Counter(seq)
    most_val = max(freq.items(), key=lambda kv: (kv[1], kv[0]))[0]
    drops = [seq[i] - seq[i + 1] for i in range(len(seq) - 1)]
    pos_sizes = sorted({d for d in drops if d > 0}, reverse=True)
    if not pos_sizes: return most_val, None
    for size in pos_sizes:
        for i, d in enumerate(drops):
            if d == size:
                after_val = seq[i + 1]
                if after_val <= cutoff: return most_val, i + 1

def composite_score(x, y):
    if len(x) == 0 or len(y) == 0: return math.nan
    delta = abs(cliff_delta(x, y)); hl = abs(hodges_lehmann(x, y))
    return delta * hl

def hodges_lehmann(x, y):
    diffs = [xi - yj for xi in x for yj in y]
    return abs(np.median(diffs))

def cliff_delta(g1, g2):
    nx, ny = len(g1), len(g2)
    greater = sum(x > y for x in g1 for y in g2)
    less    = sum(x < y for x in g1 for y in g2)
    delta = (greater - less) / (nx*ny)
    return abs(delta)

def safe_kde(x, **kwargs):
    x = np.asarray(x)
    if x.size < 2 or np.all(x == x.flat[0]): return None
    return gaussian_kde(x, **kwargs)

def compute_all_ad_pairs(groups):
    records = []
    for (i, g1), (j, g2) in combinations(enumerate(groups), 2):
        p = composite_score(g1, g2)
        records.append({'group1': i, 'group2': j, 'p_value': p})
    return pd.DataFrame(records)

def merge_with_threshold(groups, df_pairs, alpha):
    n = len(groups); parent = list(range(n))
    def find(x):
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb: parent[rb] = ra
    for _, row in df_pairs.iterrows():
        if row.p_value <= alpha: union(int(row.group1), int(row.group2))
    comps = defaultdict(list)
    for i in range(n): comps[find(i)].append(i)
    merged = []; merged_map = {}
    for merged_idx, idxs in enumerate(comps.values()):
        buf = []
        for idx in idxs: buf.extend(groups[idx])
        merged.append(buf); merged_map[merged_idx] = idxs
    return merged, merged_map

def assign_component(x, groups, mapping, original_comp):
    if any(set(sublist) == set(x) for sublist in original_comp) and len(x) > 1:
        for keys in mapping:
            if [i for i, sublist in enumerate(original_comp) if Counter(sublist) == Counter(x)][0] in mapping[keys]:
                return keys
            else:
                pass
    else:
        distance_x_avg = []
        for each_subgroup in groups:
            distance_x_avg.append(abs(np.mean(each_subgroup) - np.mean(x)))
        idx_best = distance_x_avg.index(min(distance_x_avg))
    return idx_best

def get_sample_index_in_TPM_list(Samples_Dic, TPM_values_for_gene, Factor):
    sample_index = 0
    for i in range(0, sorted(Samples_Dic.keys()).index(Factor)):
        sample_index = sample_index + len(TPM_values_for_gene[i])
    sample_span = [sample_index, sample_index+len(TPM_values_for_gene[sorted(Samples_Dic.keys()).index(Factor)])]
    return sample_span

### A4. `plot_GMM_distribution` -- verbatim, guarded by `MAKE_GMM_PLOTS`

In [ ]:
def plot_GMM_distribution(values_, batch_labels, merged_clusters_, mapping_, original_comp_, best_n_, replicate_index, max_p_value, Factor_name='', outname='GMM'):
    if not MAKE_GMM_PLOTS:
        return
    fig, axes = plt.subplots(2, 1, figsize=(9, 10), sharex=True)
    Hexcode_colors = ['#FF0000', '#FF7F00', '#FFFF00', '#00FF00', '#0000FF', '#4B0082', '#8B00FF', '#FF00FF']
    axes[0].hist(values_, bins=100, density=True, alpha=0.3, color='gray')
    for cluster_i, cluster_data in enumerate(merged_clusters_):
        cluster_data = np.array(cluster_data); kde = safe_kde(cluster_data)
        if kde is None:
            axes[0].axhline(cluster_data.flat[0], color=Hexcode_colors[cluster_i % len(Hexcode_colors)])
        else:
            xs = np.linspace(cluster_data.min(), cluster_data.max(), 2*len(cluster_data))
            weight = len(cluster_data)/len(values_)
            axes[0].plot(xs, weight*kde(xs), color=Hexcode_colors[cluster_i % len(Hexcode_colors)])
    axes[0].set_title(fr"Bottom Up Model Fit for $\it{{{Factor_name}}}$ (n_component={best_n_})")
    df_all = []; df_mean = []
    if len(replicate_index) > 0:
        for key in replicate_index:
            safe_key = key.replace('_', r'\_'); x_scatter = []; batch_scatter = []
            for j in range(replicate_index[key][0], replicate_index[key][1]):
                x_scatter.append(values_[j]); batch_scatter.append(batch_labels[j])
            if len(x_scatter) != 0:
                df_all.append(pd.DataFrame({'value': x_scatter, 'batch': batch_scatter, 'group': [fr"$\it{{{safe_key}}}$"]*len(x_scatter)}))
                df_mean.append(np.mean(x_scatter))
        df_all = [b for _, b in sorted(zip(df_mean, df_all), key=lambda x: x[0], reverse=False)]
    df_combined = pd.concat(df_all) if df_all else pd.DataFrame()
    unique_groups = df_combined['group'].unique() if len(df_combined) else []
    for i, group in enumerate(unique_groups):
        subset = df_combined[df_combined['group'] == group]; y_pos = np.full(len(subset), i + 1)
        component_label_ = assign_component(subset['value'].tolist(), merged_clusters_, mapping_, original_comp_)
        axes[1].scatter(subset['value'], y_pos, color=Hexcode_colors[component_label_ % len(Hexcode_colors)], s=45, alpha=1)
    axes[1].set_xlabel("CPM"); axes[1].set_ylabel("Cell type (pseudocells shown)")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'GMM_figures', 'AIC', '{}_{}.jpg'.format(Factor_name, outname)), dpi=300)
    plt.close()
    return

### A5. `select_and_convert_gmm_aicc` -- the core state-caller, verbatim

In [ ]:
def select_and_convert_gmm_aicc(Samples_Dic, TPM_values_for_gene, Batch_values_for_gene, Factor_name_, max_components=8):
    values = [x for sublist in TPM_values_for_gene for x in sublist]
    batch_labels = [x for sublist in Batch_values_for_gene for x in sublist]
    replicate_variability = []
    for each_replicate_values in TPM_values_for_gene:
        replicate_variability.append(0 if len(each_replicate_values) < 2 else sum(abs(x-y) for i,x in enumerate(each_replicate_values) for y in each_replicate_values[i+1:]) / (len(each_replicate_values)*(len(each_replicate_values)-1)/2))
    clustering_sensitivity = 100
    TPM_values_for_gene_cleaned = [vector for vector in TPM_values_for_gene if len(vector) > 1]
    df_pairs = compute_all_ad_pairs(TPM_values_for_gene_cleaned)
    number_of_clusters = [len(merge_with_threshold(TPM_values_for_gene_cleaned, df_pairs, test_alpha)[0]) for test_alpha in np.linspace(df_pairs['p_value'].min(), df_pairs['p_value'].max(), clustering_sensitivity, endpoint=False)]
    most_val, idx_of_elbow = find_elbow_idx_by_cutoff(number_of_clusters, 8)
    if idx_of_elbow == None:
        optimal_alpha = df_pairs['p_value'].min()
    elif most_val == 1:
        if number_of_clusters.count(1) >= len(number_of_clusters)-1:
            optimal_alpha = np.linspace(df_pairs['p_value'].min(), df_pairs['p_value'].max(), clustering_sensitivity, endpoint=False)[number_of_clusters.index(1)]
        else:
            optimal_alpha = np.linspace(df_pairs['p_value'].min(), df_pairs['p_value'].max(), clustering_sensitivity, endpoint=False)[idx_of_elbow]
    else:
        optimal_alpha = np.linspace(df_pairs['p_value'].min(), df_pairs['p_value'].max(), clustering_sensitivity, endpoint=False)[idx_of_elbow]
    optimal_clusters, optimal_mapping = merge_with_threshold(TPM_values_for_gene_cleaned, df_pairs, optimal_alpha)
    filtered_clusters = []; filtered_mapping = {}
    for new_idx, old_idx in enumerate(range(len(optimal_clusters))):
        cluster = optimal_clusters[old_idx]
        if len(cluster) > 1:
            filtered_clusters.append(cluster); filtered_mapping[len(filtered_clusters)-1] = optimal_mapping[old_idx]
    optimal_clusters = filtered_clusters
    optimal_mapping = {new_idx: filtered_mapping[old_key] for new_idx, old_key in enumerate(sorted(filtered_mapping.keys()))}
    optimal_clusters, optimal_mapping = teset_and_merge_welch(optimal_clusters, optimal_mapping, sum(sorted(replicate_variability)[-3:]) / 3)
    optimal_mapping = {new_idx: optimal_mapping[old_key] for new_idx, old_key in enumerate(sorted(optimal_mapping.keys()))}
    best_n = len(optimal_clusters)
    medians = [np.median(c) for c in optimal_clusters]
    means = [np.mean(c) for c in optimal_clusters]
    covs  = [np.std(c) for c in optimal_clusters]
    linear_clusters = [(2.0 ** np.array(c)) - 1.0 for c in optimal_clusters]
    linear_clusters = [np.clip(c, 0, None) for c in linear_clusters]
    medians_linear = [float(np.median(c)) for c in linear_clusters]
    covs_linear    = [float(np.std(c)) for c in linear_clusters]
    weights = [len(c)/len(values) for c in optimal_clusters]
    order = np.argsort(means)
    optimal_clusters = [optimal_clusters[i] for i in order]
    optimal_mapping = {new_idx: optimal_mapping[old_idx] for new_idx, old_idx in enumerate(order)}
    medians = [medians[i] for i in order]; covs = [covs[i] for i in order]
    weights = [weights[i] for i in order]; means = [means[i] for i in order]
    medians_linear = [medians_linear[i] for i in order]; covs_linear = [covs_linear[i] for i in order]
    outfile = open(_SCF, 'a')
    outfile.write(Factor_name_+'\t'+str(best_n)+'\t')
    converted_GMM_TPM = []; converted_GMM_TPM_linear = []; converted_GMM_std = []; converted_GMM_std_linear = []
    for idx, each_TPMs in enumerate(TPM_values_for_gene):
        if each_TPMs == []:
            converted_GMM_TPM.append([0]); converted_GMM_TPM_linear.append([0])
            converted_GMM_std.append([0]); converted_GMM_std_linear.append([0])
        else:
            component_label = assign_component(each_TPMs, optimal_clusters, optimal_mapping, TPM_values_for_gene_cleaned)
            converted_GMM_TPM.append(medians[component_label]); converted_GMM_TPM_linear.append(medians_linear[component_label])
            converted_GMM_std.append(covs[component_label]); converted_GMM_std_linear.append(covs_linear[component_label])
            outfile.write(sorted(Samples_Dic.keys())[idx] + ':' + str(component_label) + '\t')
    outfile.write('\n')
    replicate_index = {}
    for each in list(Samples_Dic.keys()):
        replicate_index[each] = get_sample_index_in_TPM_list(Samples_Dic, TPM_values_for_gene, each)
    pairs = list(itertools.combinations(range(len(optimal_clusters)), 2)); pvals = []; results = []; replicate_test_p = []
    if len(optimal_clusters) > 1:
        for i, j in pairs:
            t_stat, p_val = ttest_ind(optimal_clusters[i], optimal_clusters[j], equal_var=False)
            pvals.append(p_val); results.append((str(i), str(j), t_stat, p_val))
        reject, pvals_corr, _, _ = multipletests(pvals, method="holm")
        for (g1, g2, t, p), p_corr, r in zip(results, pvals_corr, reject):
            replicate_test_p.append(p_corr)
    outfile.close()
    plot_GMM_distribution(np.array(values), batch_labels, optimal_clusters, optimal_mapping, TPM_values_for_gene_cleaned, best_n, replicate_index, float(max(replicate_test_p)) if replicate_test_p else math.nan, Factor_name=Factor_name_)
    for each_i in range(0, len(converted_GMM_TPM)):
        if isinstance(converted_GMM_TPM[each_i], list):
            converted_GMM_TPM[each_i] = converted_GMM_TPM[each_i][0]
            if isinstance(converted_GMM_TPM_linear[each_i], list): converted_GMM_TPM_linear[each_i] = converted_GMM_TPM_linear[each_i][0]
            if isinstance(converted_GMM_std_linear[each_i], list): converted_GMM_std_linear[each_i] = converted_GMM_std_linear[each_i][0]
            converted_GMM_std[each_i] = converted_GMM_std[each_i][0]
        elif isinstance(converted_GMM_TPM[each_i], np.ndarray):
            converted_GMM_TPM[each_i] = converted_GMM_TPM[each_i].item(); converted_GMM_std[each_i] = converted_GMM_std[each_i].item()
    return (best_n, np.array(converted_GMM_TPM), np.array(converted_GMM_std),
            np.array(converted_GMM_TPM_linear), np.array(converted_GMM_std_linear))

print('BMMC state-calling functions loaded.')

## Part B -- Load Lambert TF list, 56 SETIA TFs, and BMMC counts

Both start from the same Lambert "Yes" TF set as GTEx. BMMC TF universe = Lambert (intersect) genes present in `adata.raw`.

In [ ]:
# Lambert confirmed TFs (same table GTEx uses)
ws  = openpyxl.load_workbook(LAMBERT_XLSX, read_only=True)['Table S1. Related to Figure 1B']
rows = list(ws.iter_rows(values_only=True))
lambert_tfs = {r[1] for r in rows[1:] if r[3] == 'Yes' and r[1] is not None}
print(f'Lambert "Yes" TFs: {len(lambert_tfs)}')

# 56 SETIA TFs
with open(SETIA56_PATH) as fh:
    setia56 = {ln.strip() for ln in fh if ln.strip()}
print(f'SETIA TFs loaded: {len(setia56)}')

In [ ]:
# Load AnnData, raw counts
adata = ad.read_h5ad(H5AD_PATH)
X_raw = adata.raw.X if adata.raw is not None else adata.X
raw_genes = pd.Index(adata.raw.var_names if adata.raw is not None else adata.var_names)
if not issparse(X_raw): X_raw = csr_matrix(X_raw)
if np.isnan(X_raw.data).any():
    X_raw.data = np.nan_to_num(X_raw.data, nan=0.0)

# BMMC TF universe = Lambert (intersect) raw genes
raw_set   = set(raw_genes)
bmmc_tfs  = sorted(lambert_tfs & raw_set)
missing   = sorted(lambert_tfs - raw_set)
print(f'Lambert Yes:        {len(lambert_tfs)}')
print(f'Present in h5ad raw: {len(bmmc_tfs)}')
print(f'Missing (filtered out upstream): {len(missing)}')

ordered_genes = bmmc_tfs                     # the TFs we run state-calling on
setia56_in    = sorted(setia56 & set(bmmc_tfs))
print(f'SETIA TFs present in BMMC universe: {len(setia56_in)} / {len(setia56)}')
miss56 = sorted(setia56 - set(bmmc_tfs))
if miss56: print(f'  SETIA TFs not in raw: {miss56}')

# OPTIONAL: how many of the missing TFs are expressed elsewhere per GTEx?
# If you have the GTEx expressed-TF list saved, set GTEX_EXPRESSED_PATH and uncomment.
# GTEX_EXPRESSED_PATH = '.../gtex_expressed_tfs.txt'
# gtex_expr = {l.strip() for l in open(GTEX_EXPRESSED_PATH)}
# print(f'Missing-in-BMMC but expressed in GTEx: {len(set(missing) & gtex_expr)} / {len(missing)}')

In [ ]:
# Build pseudocells per cell type (BMMC params), over ordered_genes
gene_idx  = [raw_genes.get_loc(g) for g in ordered_genes]
X_raw_sub = X_raw[:, gene_idx]
celltypes = adata.obs[CELLTYPE_COL].astype(str).values

# validate sister-group labels exist
all_grouped = [c for v in SISTER_GROUPS.values() for c in v]
present_cts = set(celltypes)
bad = [c for c in all_grouped if c not in present_cts]
if bad:
    print(f'WARNING: these SISTER_GROUPS labels are NOT in {CELLTYPE_COL}: {bad}')
    print(f'Available cell types: {sorted(present_cts)}')

use_cts = sorted(c for c in present_cts if c not in EXCLUDE_CELLTYPES)
print(f'Cell types used: {len(use_cts)} (excluded {EXCLUDE_CELLTYPES})')

rng = np.random.default_rng(RANDOM_SEED)
Samples_Dic = {}; mRNA_steady_states = {}; pseudocell_info = []
for ct in use_cts:
    mask = np.where(celltypes == ct)[0]; n_cells = len(mask)
    if n_cells < MIN_CELLS_PER_PSEUDOCELL * MIN_PSEUDOCELLS:
        if n_cells >= 2 * 10: K = MIN_PSEUDOCELLS
        else:
            print(f'  SKIP {ct}: only {n_cells} cells'); continue
    else:
        K = min(MAX_PSEUDOCELLS_PER_CELLTYPE, n_cells // MIN_CELLS_PER_PSEUDOCELL)
    K = max(K, MIN_PSEUDOCELLS)
    shuffled = rng.permutation(mask); chunks = np.array_split(shuffled, K)
    pcs = []
    for i, chunk in enumerate(chunks):
        pid = f'{ct}_pseudo_{i:03d}'
        counts = np.asarray(X_raw_sub[chunk].sum(axis=0)).flatten()
        lib    = float(np.asarray(X_raw[chunk].sum()).flatten()[0])
        if lib <= 0: continue
        cpm = counts / lib * 1e6
        mRNA_steady_states[pid] = {g: float(cpm[gi]) for gi, g in enumerate(ordered_genes)}
        pcs.append(pid); pseudocell_info.append({'cell_type': ct, 'pseudocell': pid, 'n_cells': len(chunk), 'lib_size': lib})
    Samples_Dic[ct] = pcs

info_df = pd.DataFrame(pseudocell_info)
print(f'\nBuilt {len(mRNA_steady_states)} pseudocells across {len(Samples_Dic)} cell types')
print(info_df.groupby("cell_type").size().to_string())

## Part C -- Run BMMC state-caller over all TFs

Same loop structure as `build_setia_input`, but over `ordered_genes` (the ~1171 Lambert TFs). For each gene we keep `n_states`, the per-state medians (log2), and the per-cell-type assigned state.

In [ ]:
def gene_value(pseudocell_id, gene):
    v = mRNA_steady_states[pseudocell_id].get(gene, 0.0)
    return float(np.log2(v + 1)) if USE_LOG2 else float(v)

cts_sorted = sorted(Samples_Dic.keys())

state_results = {}     # gene -> {'n_states', 'medians_log', 'tissue_states'}
n_states_dict = {}
skipped = []
t0 = time.time()
for col_idx, gene in enumerate(ordered_genes):
    TPM_values_for_gene = []; Batch_values_for_gene = []
    for ct in cts_sorted:
        vals = [gene_value(p, gene) for p in Samples_Dic[ct]]
        bats = [p.split('_')[2] if len(p.split('_')) > 2 else 'b1' for p in Samples_Dic[ct]]
        TPM_values_for_gene.append(vals); Batch_values_for_gene.append(bats)
    try:
        best_n, conv, std, conv_lin, std_lin = select_and_convert_gmm_aicc(
            Samples_Dic, TPM_values_for_gene, Batch_values_for_gene, gene)
        conv = np.asarray(conv, dtype=float)
        med_log = sorted({round(float(v), 6) for v in conv})          # realized state medians (log2)
        idx_of  = {m: i for i, m in enumerate(med_log)}
        tissue_states = {ct: idx_of[round(float(conv[k]), 6)] for k, ct in enumerate(cts_sorted)}
        state_results[gene] = {'n_states': int(best_n), 'medians_log': med_log, 'tissue_states': tissue_states}
        n_states_dict[gene] = int(best_n)
    except Exception as e:
        skipped.append((gene, str(e)))
    if (col_idx + 1) % 25 == 0:
        el = time.time() - t0; eta = el / (col_idx+1) * (len(ordered_genes) - col_idx - 1)
        print(f'  {col_idx+1}/{len(ordered_genes)}  ({el:.0f}s, ETA {eta:.0f}s)  skipped={len(skipped)}')
print(f'Done in {time.time()-t0:.0f}s. States called for {len(state_results)} TFs; skipped {len(skipped)}.')
if skipped: print('First few skipped:', skipped[:5])

dist = Counter(v['n_states'] for v in state_results.values())
for n, c in sorted(dist.items()):
    print(f'  {n} states: {c} TFs ({c/len(state_results)*100:.1f}%)')

In [ ]:
# consistency: per cell type, fraction of pseudocells whose nearest state median == assigned state
def compute_consistency(genes, sr, Samples_Dic, mRNA_ss, cts_sorted):
    records = []
    for gene in genes:
        res = sr[gene]; n = res['n_states']; med = res['medians_log']; ts = res['tissue_states']
        cons = []
        for ct in cts_sorted:
            assigned = ts.get(ct)
            if assigned is None: continue
            vals = [np.log2(float(mRNA_ss[p].get(gene, 0.0)) + 1) for p in Samples_Dic[ct]]
            if not vals: continue
            if n < 2 or len(med) < 2:
                cons.append(1.0)
            else:
                ok = sum(int(np.argmin([abs(v - m) for m in med])) == assigned for v in vals)
                cons.append(ok / len(vals))
        records.append({'gene': gene, 'n_states': n, 'mean_consistency': np.mean(cons) if cons else np.nan})
    return pd.DataFrame(records)

all_wc = compute_consistency(list(state_results.keys()), state_results, Samples_Dic, mRNA_steady_states, cts_sorted).dropna(subset=['mean_consistency'])
print(f'TFs with consistency: {len(all_wc)}')
print(f'  n_states=1:  {(all_wc.n_states==1).sum()}')
print(f'  n_states>=2: {(all_wc.n_states>=2).sum()}')
print(all_wc.mean_consistency.describe().round(3))

## Part D -- Panel 1: n_states distribution (pie)

In [ ]:
mpl.rcParams.update({'font.family':'Arial','font.size':7,'axes.linewidth':0.8,
    'xtick.major.width':0.8,'ytick.major.width':0.8,'pdf.fonttype':42,'ps.fonttype':42})

ns = all_wc['n_states'].value_counts().sort_index()
total = len(all_wc); multi = int((all_wc.n_states>=2).sum()); mean_st = all_wc.n_states.mean()
print(f'Expressed TFs: {total} | multi-state: {multi} ({multi/total*100:.1f}%) | mean n_states: {mean_st:.2f}')

fig, ax = plt.subplots(figsize=(5,5))
states = ns.index.tolist(); counts = ns.values.tolist()
base = ['#AAAAAA'] + [plt.cm.Blues(0.3 + 0.7*i/max(1,(len(states)-2))) for i in range(len(states)-1)]
ax.pie(counts, labels=[f'{s} state{"s" if s>1 else ""}' for s in states], colors=base,
       autopct=lambda p: f'{p:.1f}%' if p>=3 else '', startangle=90,
       wedgeprops=dict(edgecolor='white', linewidth=1.2), pctdistance=0.75)
ax.set_title(f'Discrete expression states across {total} expressed TFs\n'
             f'(BMMC single-cell; Lambert start 1639, detected {total})\n'
             f'{multi/total*100:.1f}% multi-state | mean = {mean_st:.1f}', fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUTDIR}/Fig3_BMMC_nstates_pie.pdf', bbox_inches='tight')
plt.savefig(f'{OUTDIR}/Fig3_BMMC_nstates_pie.png', dpi=300, bbox_inches='tight')
plt.show(); print(ns)

## Part E -- Panel 2: consistency violin (56 SETIA TFs vs other) + n_states-matched check

The violin is the panel. Below it we PRINT an n_states-matched check (no ranking, no extra figure): if SETIA TFs differ from background in n_states and consistency tracks n_states, the raw violin gap could be partly an n_states artifact. We report the within-stratum percentile comparison so the violin claim is clean.

In [ ]:
wc_multi   = all_wc[all_wc.n_states>=2].copy()
is_setia   = wc_multi.gene.isin(setia56_in)
setia_cons = wc_multi[is_setia]['mean_consistency']
other_cons = wc_multi[~is_setia]['mean_consistency']
_, p = mannwhitneyu(setia_cons, other_cons, alternative='greater')

print(f'n_states>=2: {len(wc_multi)} TFs')
print(f'SETIA TFs (n={len(setia_cons)}): mean={setia_cons.mean():.3f}, median={setia_cons.median():.3f}')
print(f'Other TFs (n={len(other_cons)}): mean={other_cons.mean():.3f}, median={other_cons.median():.3f}')
print(f'Mann-Whitney (SETIA > other) p = {p:.4e}')

fig, ax = plt.subplots(figsize=(7.5, 9.0))
cM, cO = '#D55E00', '#0072B2'
parts = ax.violinplot([setia_cons.values, other_cons.values], positions=[0,1],
                      showmedians=True, showextrema=True, widths=0.7)
parts['bodies'][0].set_facecolor(cM); parts['bodies'][1].set_facecolor(cO)
for pc in parts['bodies']: pc.set_alpha(0.65); pc.set_edgecolor('none')
parts['cmedians'].set_color('white'); parts['cmedians'].set_linewidth(1.2)
for k in ('cmaxes','cmins','cbars'): parts[k].set_color('#555'); parts[k].set_linewidth(0.8)
ax.set_xticks([0,1]); ax.set_xticklabels([f'SETIA 56\n(n={len(setia_cons)})', f'other TFs\n(n={len(other_cons)})'])
ax.set_ylabel('within-cell-type consistency'); ax.set_ylim(0,1.02)
ax.set_title(f'BMMC (Mann-Whitney p = {p:.1e})', fontsize=8)
for s in ('top','right'): ax.spines[s].set_visible(False)

# Individual data points — both groups
np.random.seed(42)
# SETIA 56: larger, more opaque
jitter_s = np.random.uniform(-0.08, 0.08, len(setia_cons))
ax.scatter(jitter_s, setia_cons.values,
           color='#8B3A00', s=6, alpha=0.6, zorder=4, edgecolors='none')
# Other TFs: smaller, more transparent
jitter_o = np.random.uniform(-0.08, 0.08, len(other_cons))
ax.scatter(1 + jitter_o, other_cons.values,
           color='#003D5B', s=1.5, alpha=0.12, zorder=4, edgecolors='none')
ax.yaxis.grid(True, alpha=0.25, linestyle='-', linewidth=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(f'{OUTDIR}/Fig3_BMMC_consistency_violin.pdf', bbox_inches='tight')
plt.savefig(f'{OUTDIR}/Fig3_BMMC_consistency_violin.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ---- n_states-matched check (printed only; not a panel, not a ranking) ----
print('='*56); print('n_states-matched check on the violin'); print('='*56)
rho, prho = spearmanr(wc_multi.n_states, wc_multi.mean_consistency)
print(f'Spearman(n_states, consistency) = {rho:+.3f}  p={prho:.1e}')
print(f'n_states  SETIA median={wc_multi[is_setia].n_states.median():.1f}  '
      f'| other median={wc_multi[~is_setia].n_states.median():.1f}')
# within-stratum percentile of consistency, then compare SETIA vs other
wc_multi['pct'] = wc_multi.groupby('n_states')['mean_consistency'].rank(pct=True)
mp = wc_multi[is_setia]['pct']; op = wc_multi[~is_setia]['pct']
_, pp = mannwhitneyu(mp, op, alternative='greater')
print(f'within-n_states percentile  SETIA mean={mp.mean():.3f}  | other mean={op.mean():.3f}  '
      f'(MWU SETIA>other p={pp:.3e})')
print('-> if the percentile gap and p stay in the same direction as the raw violin,')
print('   the SETIA-vs-other difference is not an n_states artifact.')

## Part F -- Panel 3: Hamming coherence per sister-subtype group + Control

State matrix (cell type x informative TF) -> Hamming. For each sister-subtype group, within/between Hamming ratio (lower = more coherent). Control = shuffled fake group (one cell type sampled per group, 1000 draws).

In [ ]:
GROUP_COLORS = {'Erythroid':'#F4D27A','Monocyte':'#7C6FA8','GMP':'#8B2C3A','CLP':'#D597B6',
                'DC':'#C8455A','B':'#A57F4B','NaiveT_CD4':'#E89B7B','Cytotoxic':'#A04848','Control':'#BBBBBB'}

# sort by ratio descending → most coherent (lowest ratio) ends up at top
rdf = pd.DataFrame(
    [{'group': g, 'ratio': r, 'color': GROUP_COLORS.get(g, '#AAAAAA')} for g, r in ratios.items()]
).sort_values('ratio', ascending=False)

fig, ax = plt.subplots(figsize=(4.5*1.3, 3.5))
bars = ax.barh(
    rdf['group'], rdf['ratio'],
    color=rdf['color'], edgecolor='black', linewidth=0.4, height=0.6
)

max_ratio = rdf['ratio'].max()
ax.set_xlabel('Within-group Hamming / Between-group Hamming', fontsize=11)
ax.set_title('BMMC sister-subtypes cluster by TF state similarity', fontsize=11)
ax.set_xlim(0, max_ratio * 1.1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig(f'{OUTDIR}/Fig3_BMMC_hamming_ratio.pdf', bbox_inches='tight')
plt.savefig(f'{OUTDIR}/Fig3_BMMC_hamming_ratio.png', dpi=300, bbox_inches='tight')
plt.show()

for g, r in sorted(ratios.items(), key=lambda kv: kv[1]): print(f'  {g:<12}: {r:.3f}')

## Part G -- Summary numbers for paper

In [ ]:
print('='*56); print('BMMC FIGURE 3 -- NUMBERS'); print('='*56)
print(f'Lambert start:              {len(lambert_tfs)}')
print(f'Detected in BMMC (raw):     {len(ordered_genes)}')
print(f'State-called:               {len(state_results)}')
print(f'Multi-state (>=2):          {multi} ({multi/total*100:.1f}%)')
print(f'Mean n_states:              {mean_st:.2f}')
print(f'SETIA TFs (n_states>=2):    n={len(setia_cons)}, mean cons={setia_cons.mean():.3f}')
print(f'Other TFs  (n_states>=2):   n={len(other_cons)}, mean cons={other_cons.mean():.3f}')
print(f'Mann-Whitney p:             {p:.3e}')
print('Sister-subtype Hamming ratios:')
for g, r in sorted(ratios.items(), key=lambda kv: kv[1]): print(f'  {g:<12}: {r:.3f}')

In [ ]:
# ===========================================================================
# Export SETIA-style discrete-state output (same convention as build_setia_input)
#   Steady_state_count.txt   gene<TAB>n_states<TAB>celltype:state_label ...
#   discrete_states.txt       rows=cell type, cols=gene, value=log2 state median
#   discrete_states_std.txt   rows=cell type, cols=gene, value=log2 std of assigned state
# Rebuilt from state_results; state-calling functions untouched.
# ===========================================================================
GRN_OUT = OUTDIR                    # bmmc_tf_output; change if desired
os.makedirs(GRN_OUT, exist_ok=True)

samples_sorted = cts_sorted
all_genes = [g for g in ordered_genes if g in state_results]
med_mat, std_mat, ssc_lines = {}, {}, []

for gene in all_genes:
    res     = state_results[gene]
    n       = int(res['n_states'])
    med_log = list(res['medians_log'])         # sorted log2 medians, len == n_states
    ts      = res['tissue_states']
    nstate  = max(n, len(med_log))

    # group log2 pseudocell values by the FINAL assigned state label
    state_vals = [[] for _ in range(nstate)]
    for ct in samples_sorted:
        si = ts.get(ct)
        if si is None or not (0 <= si < nstate):
            continue
        for p in Samples_Dic[ct]:
            v = np.log2(float(mRNA_steady_states[p].get(gene, 0.0)) + 1)
            state_vals[si].append(v)
    std_log = [float(np.std(sv)) if sv else 0.0 for sv in state_vals]

    med_row, std_row, labels = {}, {}, []
    for ct in samples_sorted:
        si = ts.get(ct, 0)
        if not (0 <= si < nstate):
            si = 0
        med_row[ct] = float(med_log[si]) if si < len(med_log) else (float(med_log[0]) if med_log else 0.0)
        std_row[ct] = std_log[si] if si < len(std_log) else 0.0
        labels.append(f'{ct}:{si}')

    med_mat[gene] = med_row
    std_mat[gene] = std_row
    ssc_lines.append(f'{gene}\t{n}\t' + '\t'.join(labels))

ds   = pd.DataFrame(med_mat, index=samples_sorted).reindex(columns=all_genes)
dstd = pd.DataFrame(std_mat, index=samples_sorted).reindex(columns=all_genes)
ds.index.name = 'cell_type'
dstd.index.name = 'cell_type'

ds.to_csv(f'{GRN_OUT}/discrete_states.txt',     sep='\t', float_format='%.4f')
dstd.to_csv(f'{GRN_OUT}/discrete_states_std.txt', sep='\t', float_format='%.4f')
with open(f'{GRN_OUT}/Steady_state_count.txt', 'w') as fh:
    fh.write('\n'.join(ssc_lines) + '\n')

print(f'BMMC SETIA output written to {GRN_OUT}/')
print(f'  genes: {len(all_genes)}   cell types: {len(samples_sorted)}')
print(f'  discrete_states.txt     {ds.shape}')
print(f'  discrete_states_std.txt {dstd.shape}')
print(f'  Steady_state_count.txt  {len(ssc_lines)} lines')
